<a href="https://colab.research.google.com/github/gdhameja1/ML/blob/main/Groceries_MarketBasket_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Market Basket Analysis on Groceries_MarketBasket.csv

This notebook performs Market Basket Analysis (Apriori + association rules) on `Groceries_MarketBasket.csv` (wide format: one row per transaction, columns `TransactionID`, `Item_1`, `Item_2`, ...).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Market Basket Analysis on Groceries_MarketBasket.csv
# ----------------------------------------------------

%pip install pandas mlxtend

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [ ]:
# -----------------------------
# Parameters you can tweak
# -----------------------------
FILE_PATH = "Groceries_MarketBasket.csv"

MIN_SUPPORT = 0.02       # e.g., itemsets in at least 2% of all transactions
MIN_CONFIDENCE = 0.1     # minimum confidence for rules
MIN_LIFT = 1.0           # minimum lift for rules
MAX_ITEMSET_LENGTH = 3   # maximum size of itemsets (1 = singles, 2 = pairs, 3 = triples, etc.)
TOP_N_TO_PRINT = 20      # how many top itemsets / rules to print

In [ ]:
df = pd.read_csv('Groceries_MarketBasket.csv')

In [ ]:
# -----------------------------
# 1. Load the data
# -----------------------------
df = pd.read_csv("Groceries_MarketBasket.csv")

# Identify the item columns (Item_1, Item_2, ...)
item_cols = [c for c in df.columns if c.lower().startswith("item_")]

In [ ]:
# -----------------------------
# 2. Convert to list-of-lists
#    format for each basket
# -----------------------------
transactions = []
for _, row in df[item_cols].iterrows():
    # Take all non-empty, non-NaN items in this row
    basket = [
        str(it).strip()
        for it in row.values
        if pd.notna(it) and str(it).strip() != ""
    ]
    transactions.append(basket)

print(f"Loaded {len(transactions)} transactions.")

Loaded 1000 transactions.


In [ ]:
# -----------------------------
# 3. One-hot encode baskets
# -----------------------------
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket_encoded = pd.DataFrame(te_array, columns=te.columns_)

print(f"Number of unique items: {basket_encoded.shape[1]}")

Number of unique items: 80


In [ ]:
# -----------------------------
# 4. Find frequent itemsets
# -----------------------------
frequent_itemsets = apriori(
    basket_encoded,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_ITEMSET_LENGTH
)

# Sort by support (most frequent first)
frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False).reset_index(drop=True)

print("=== Top Frequent Itemsets ===")

def fmt_itemset(itemset):
    return ", ".join(sorted(list(itemset)))

for idx, row in frequent_itemsets.head(TOP_N_TO_PRINT).iterrows():
    print(f"{idx+1:2d}. {{{fmt_itemset(row['itemsets'])}}} "
          f" | support = {row['support']:.3f}")

=== Top Frequent Itemsets ===
 1. {beef}  | support = 0.110
 2. {ice_cream}  | support = 0.106
 3. {shredded_cheese}  | support = 0.104
 4. {sugar}  | support = 0.104
 5. {spinach}  | support = 0.104
 6. {yogurt_drink}  | support = 0.103
 7. {onion}  | support = 0.103
 8. {frozen_vegetables}  | support = 0.102
 9. {blueberry}  | support = 0.101
10. {granola_bar}  | support = 0.101
11. {toothpaste}  | support = 0.101
12. {garlic}  | support = 0.099
13. {shampoo}  | support = 0.098
14. {protein_bar}  | support = 0.098
15. {noodles}  | support = 0.098
16. {yogurt}  | support = 0.098
17. {corn}  | support = 0.097
18. {mushroom}  | support = 0.097
19. {plastic_wrap}  | support = 0.096
20. {ginger}  | support = 0.093


In [ ]:
# -----------------------------
# 5. Generate association rules
# -----------------------------
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

# Filter by lift
rules = rules[rules["lift"] >= MIN_LIFT].copy()

# Sort rules by lift (then confidence)
rules = rules.sort_values(["lift", "confidence"], ascending=False).reset_index(drop=True)

#print(rules)
print("=== Top Association Rules ===")

for idx, row in rules.head(TOP_N_TO_PRINT).iterrows():
    antecedents = fmt_itemset(row['antecedents'])
    consequents = fmt_itemset(row['consequents'])

print(
        f"{idx+1:2d}. {{{antecedents}}}  ->  {{{consequents}}}  | "
        f"support = {row['support']:.3f}, "
        f"confidence = {row['confidence']:.3f}, "
        f"lift = {row['lift']:.3f}"
    )

=== Top Association Rules ===
 2. {toothpaste}  ->  {chips}  | support = 0.020, confidence = 0.198, lift = 2.330


In [ ]:
# -----------------------------
# 6. (Optional) Save to CSV
# -----------------------------
frequent_itemsets.to_csv("MBA_Frequent_Itemsets.csv", index=False)
rules.to_csv("MBA_Association_Rules.csv", index=False)

print("Saved:")
print(" - MBA_Frequent_Itemsets.csv")
print(" - MBA_Association_Rules.csv")
print("Done.")

Saved:
 - MBA_Frequent_Itemsets.csv
 - MBA_Association_Rules.csv
Done.


# Market Basket Analysis: Interpretation Guide

Market Basket Analysis uses three core metrics to evaluate the strength and relevance of association rules (e.g., {Bread} -> {Butter}).

## 1. Support (How popular is the itemset?)
Definition: The percentage of all transactions that contain the specific combination of items.
Formula: (Transactions with both A and B) / (Total Transactions)
Business Meaning: It tells you how often the rule occurs. If support is very low, the rule might be a statistical fluke or not worth a major marketing investment because it doesn't happen often enough to move the needle.

## 2. Confidence (How reliable is the prediction?)
Definition: Given that item A is in the basket, what is the probability that item B is also there?
Formula: Support(A, B) / Support(A)
Business Meaning: It measures the "strength" of the association. A confidence of 0.75 (75%) means that in 3 out of 4 times someone bought A, they also bought B. High confidence is great for cross-selling strategies.

## 3. Lift (Is this a real relationship or just a coincidence?)
Definition: The ratio of the observed support to that expected if A and B were independent.
Formula: Confidence(A -> B) / Support(B)
Business Meaning: This is often the most important metric.
Lift > 1: Positive correlation. Buying A increases the likelihood of buying B. The higher the lift, the stronger the "link."
Lift = 1: No correlation. A and B are bought together purely by chance (e.g., everyone buys Milk, so it appears in many baskets regardless of what else is there).
Lift < 1: Negative correlation. Buying A actually decreases the likelihood of buying B (they might be substitutes, like two different brands of the same product).


## Summary for Decision Making:
    
High Support + High Confidence: These are your "Bread and Butter" rules. Use them for store layout (place them far apart to force customers to walk through the store).
Low Support + High Lift: These are "Niche Opportunities." These items are rarely bought, but when they are, they are almost always bought together. Great for targeted bundle offers or "Customers who bought this also liked..." recommendations.